# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nnanwubeikenna-prog/ikenna-flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Decision-Support Action Queue & Reason Codes**
* In this cross-sectional dataset (restricted to published, non-deleted content), items are ranked for human review prioritization.
* **Reason Codes:**
  * `STALE_HIGH_DEMAND`: Pages with substantial search volume that have not been updated in over 12 months.
  * `THIN_HIGH_DEMAND`: Pages with search volume > 1,000 but containing fewer than 800 words.
  * `ROUTINE_AUDIT`: Pages with steady volume and recent update timestamps.
* **Action Labels:** `HIGH_PRIORITY_REFRESH`, `EXPAND_CONTENT`, and `MONITOR_STEADY`.

In [1]:
import os
import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
con.execute("CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')")

df_queue = con.sql("""
SELECT
    content_hash_id,
    client_hash_id,
    content_type,
    COALESCE(search_volume, 0) AS search_volume,
    COALESCE(word_count, 0) AS word_count,
    COALESCE(backlinks, 0) AS backlinks,
    content_updated_date,
    (
        COALESCE(LN(search_volume + 1), 0) * 1.5 +
        CASE WHEN content_updated_date < DATE '2025-06-01' OR content_updated_date IS NULL THEN 3.0 ELSE 0.0 END +
        CASE WHEN word_count < 800 OR word_count IS NULL THEN 2.0 ELSE 0.0 END
    ) AS priority_score,
    CASE
        WHEN (content_updated_date < DATE '2025-06-01' OR content_updated_date IS NULL) AND search_volume > 1000 THEN 'STALE_HIGH_DEMAND'
        WHEN (word_count < 800 OR word_count IS NULL) AND search_volume > 1000 THEN 'THIN_HIGH_DEMAND'
        ELSE 'ROUTINE_AUDIT'
    END AS reason_code,
    CASE
        WHEN (content_updated_date < DATE '2025-06-01' OR content_updated_date IS NULL) AND search_volume > 1000 THEN 'HIGH_PRIORITY_REFRESH'
        WHEN (word_count < 800 OR word_count IS NULL) AND search_volume > 1000 THEN 'EXPAND_CONTENT'
        ELSE 'MONITOR_STEADY'
    END AS action_label
FROM dim_content
WHERE is_published = TRUE AND is_deleted = FALSE
ORDER BY priority_score DESC;
""").df()

print("--- Top 5 Scored Playbook Items ---")
display(df_queue.head(5)[['content_hash_id', 'client_hash_id', 'search_volume', 'word_count', 'priority_score', 'reason_code', 'action_label']])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Top 5 Scored Playbook Items ---


,content_hash_id,client_hash_id,search_volume,word_count,priority_score,reason_code,action_label
0,content_b9ffa30eb293951f,client_73cda7b4e4f265ea,368000,0,21.223761,THIN_HIGH_DEMAND,EXPAND_CONTENT
1,content_ac0c525eb243379b,client_fef1a8f436438636,246000,0,20.619636,THIN_HIGH_DEMAND,EXPAND_CONTENT
2,content_621e4dc78b849ce4,client_73cda7b4e4f265ea,201000,0,20.316598,THIN_HIGH_DEMAND,EXPAND_CONTENT
3,content_a31c400b511b1458,client_fef1a8f436438636,201000,0,20.316598,THIN_HIGH_DEMAND,EXPAND_CONTENT
4,content_0184167e6037fbc7,client_fef1a8f436438636,201000,0,20.316598,THIN_HIGH_DEMAND,EXPAND_CONTENT


**Intended Use and Operational Boundaries**
* **Intended Use:** Serves strictly as an editorial triage tool to highlight URLs that warrant human inspection first.
* **Dataset & Method Limits:**
  * Observational only: Ranking does not guarantee traffic increases upon revision.
  * Survivorship: Analysis filters out deleted URLs, observing only surviving assets.
  * Contextual limits: Does not account for unmeasured offline market trends or brand-specific navigation queries.

In [2]:
action_summary = df_queue['action_label'].value_counts().reset_index()
action_summary.columns = ['Action Label', 'Item Count']
action_summary['Share (%)'] = round((action_summary['Item Count'] / len(df_queue)) * 100, 2)
print("Action Allocation Summary:")
display(action_summary)

Action Allocation Summary:


,Action Label,Item Count,Share (%)
0,MONITOR_STEADY,409561,99.52
1,EXPAND_CONTENT,1979,0.48


**Human-in-the-Loop Safeguards & Automation No-Go List**
* **Required Review:** Human editors must verify search intent match and check for recent offline product changes before executing rewrites.
* **Strict Automation No-Go List:**
  * No automated bulk text rewriting or direct CMS publishing.
  * No automated URL redirects or page deletions.
  * No automated modification of core brand or policy pages.

In [3]:
review_queue = df_queue[df_queue['action_label'].isin(['HIGH_PRIORITY_REFRESH', 'EXPAND_CONTENT'])].head(10)
print(f"Sample Queue Requiring Human Verification (N={len(review_queue)}):")
display(review_queue[['content_hash_id', 'client_hash_id', 'search_volume', 'word_count', 'reason_code', 'action_label']])

Sample Queue Requiring Human Verification (N=10):


,content_hash_id,client_hash_id,search_volume,word_count,reason_code,action_label
0,content_b9ffa30eb293951f,client_73cda7b4e4f265ea,368000,0,THIN_HIGH_DEMAND,EXPAND_CONTENT
1,content_ac0c525eb243379b,client_fef1a8f436438636,246000,0,THIN_HIGH_DEMAND,EXPAND_CONTENT
2,content_621e4dc78b849ce4,client_73cda7b4e4f265ea,201000,0,THIN_HIGH_DEMAND,EXPAND_CONTENT
3,content_a31c400b511b1458,client_fef1a8f436438636,201000,0,THIN_HIGH_DEMAND,EXPAND_CONTENT
4,content_0184167e6037fbc7,client_fef1a8f436438636,201000,0,THIN_HIGH_DEMAND,EXPAND_CONTENT
5,content_7e6779733b1dd409,client_fef1a8f436438636,165000,0,THIN_HIGH_DEMAND,EXPAND_CONTENT
6,content_ff42f4a65f10744c,client_fef1a8f436438636,135000,0,THIN_HIGH_DEMAND,EXPAND_CONTENT
7,content_fce8d3674aba5148,client_08a6a72ff48e62c0,110000,0,THIN_HIGH_DEMAND,EXPAND_CONTENT
8,content_9755ef5214465568,client_fef1a8f436438636,110000,0,THIN_HIGH_DEMAND,EXPAND_CONTENT
9,content_6011e836cf18643a,client_3ffa76342f366962,110000,0,THIN_HIGH_DEMAND,EXPAND_CONTENT


**Performance Tracking and Retraining Triggers**
* **Signal Drift:** Trigger model review if the proportion of actionable items deviates by >25% month-over-month.
* **Retrain Schedule:** Re-evaluate rankings quarterly or following major search algorithm baseline shifts.

In [4]:
drift_baseline = {
    "total_published_assets": int(len(df_queue)),
    "actionable_assets_count": int(len(df_queue[df_queue['action_label'] != 'MONITOR_STEADY'])),
    "actionable_rate": round(float((df_queue['action_label'] != 'MONITOR_STEADY').mean()), 4),
    "median_search_volume_flagged": float(df_queue[df_queue['action_label'] != 'MONITOR_STEADY']['search_volume'].median())
}
print("Operational Monitoring Baseline:")
print(json.dumps(drift_baseline, indent=2))

Operational Monitoring Baseline:
{
  "total_published_assets": 411540,
  "actionable_assets_count": 1979,
  "actionable_rate": 0.0048,
  "median_search_volume_flagged": 2400.0
}


**Artifact Generation for Research Paper**
* Scored queue exported to `work/outputs/action_playbook_queue.csv`.
* Summary figures and monitoring metrics exported to `work/figures/`.

In [5]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Export Scored CSV
df_queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)
print("Saved: work/outputs/action_playbook_queue.csv")

# 2. Export Metrics JSON
with open("work/figures/playbook_summary.json", "w") as f:
    json.dump(drift_baseline, f, indent=2)
print("Saved: work/figures/playbook_summary.json")

# 3. Export Summary Figure
fig, ax = plt.subplots(figsize=(8, 4))
df_queue['action_label'].value_counts().plot(kind='barh', color=['#1f77b4', '#ff7f0e', '#2ca02c'], ax=ax)
ax.set_title("Operational Action Distribution (dim_content)")
ax.set_xlabel("Number of URLs")
plt.tight_layout()
plt.savefig("work/figures/playbook_action_distribution.png", dpi=150)
plt.close()
print("Saved: work/figures/playbook_action_distribution.png")

Saved: work/outputs/action_playbook_queue.csv
Saved: work/figures/playbook_summary.json
Saved: work/figures/playbook_action_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.